In [3]:
from typing import List, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
import numpy as np

In [4]:
test_corpus = ["minecraft redstone farm", "python programming tutorial"]
vec = TfidfVectorizer()
res = vec.fit_transform(test_corpus)

print("Type of fit_transform result:", type(res))
print("Array shape:", res.toarray().shape)

Type of fit_transform result: <class 'scipy.sparse._csr.csr_matrix'>
Array shape: (2, 6)


In [5]:
def group_playlist_by_tfidf(playlist_videos: List[Dict], distance_threshold: float=0.7)->Dict[str, List[Dict]]:
    """Clusters playlist videos using TF-IDF feature extraction and agglomerative cosine distance clustering."""
    if not playlist_videos:
        return {}
    
    # Prepare raw text corpus for each video
    corpus = []
    for v in playlist_videos:
        title = v.get("title", "")
        desc = v.get("description", "")
        tags = " ".join(v.get("tags") or [])
        full_text = f"{title} {desc} {tags}".strip()
        corpus.append(full_text)
        
    if not any(corpus):
        return {"all_videos": playlist_videos}
        
    # Extract TF-IDF matrix
    vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
    tfidf_sparse = vectorizer.fit_transform(corpus)
    tfidf_matrix = np.asarray(tfidf_sparse.todense())
    feature_names = np.array(vectorizer.get_feature_names_out())
    
    # Perform agglomerative clustering with cosine distance
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=distance_threshold
    )
    labels = clustering.fit_predict(tfidf_matrix)
    
    clusters = {}
    for idx, label in enumerate(labels):
        clusters.setdefault(label, []).append((playlist_videos[idx], tfidf_matrix[idx]))
    topic_groups = {}
    for label, cluster_items in clusters.items():
        vids = [item[0] for item in cluster_items]
        vectors = np.array([item[1] for item in cluster_items])
        mean_vector = vectors.mean(axis=0)
        top_indices = mean_vector.argsort()[-2:][::-1]
        top_words = feature_names[top_indices]
        auto_label = '_'.join(top_words) if len(top_words) > 0 else f"cluster_{label}"
        topic_groups[auto_label] = vids
    return topic_groups

In [6]:
import sys
sys.path.insert(0, "../src")
from youtube_client import YouTubeClient
from topic_analysis import group_playlist_by_graph_jaccard
from candidate_retrieval import generate_topic_search_queries, filter_existing_playlist_videos
from ranking import rank_candidates

playlist_url = "https://www.youtube.com/watch?v=2xcFM9CBiOE&list=PLIdGxYqxOZEXyzTG_WXp9cujA3_xgp5Ps"
client = YouTubeClient()
print("Fetching playlist videos...")
playlist_videos = client.fetch_playlist_videos(playlist_url)
# Assuming playlist_videos is loaded from your YouTubeClient
print("=== 1. Graph-Based Jaccard Clustering ===")
graph_groups = group_playlist_by_graph_jaccard(playlist_videos, similarity_threshold=0.08)
for topic, vids in graph_groups.items():
    print(f"Topic '{topic}': {len(vids)} videos")

graph_queries = generate_topic_search_queries(graph_groups)
print("Generated Search Queries:", graph_queries)

print("\n" + "="*50 + "\n")

print("=== 2. TF-IDF + Cosine Clustering ===")
tfidf_groups = group_playlist_by_tfidf(playlist_videos, distance_threshold=0.7)
for topic, vids in tfidf_groups.items():
    print(f"Topic '{topic}': {len(vids)} videos")

tfidf_queries = generate_topic_search_queries(tfidf_groups)
print("Generated Search Queries:", tfidf_queries)

Fetching playlist videos...
=== 1. Graph-Based Jaccard Clustering ===
Topic 'lyrics_website': 42 videos
Topic 'lyric_forrest': 2 videos
Topic 'new_support': 2 videos
Topic 'believe_nazare': 1 videos
Topic 'triumphant_power': 1 videos
Topic 'much_live': 1 videos
Topic 'realest_who': 1 videos
Generated Search Queries: {'lyrics_website': ['lyrics website worship'], 'lyric_forrest': ['lyric forrest hiforrest'], 'new_support': ['new support only'], 'believe_nazare': ['believe nazare silas'], 'triumphant_power': ['triumphant power song'], 'much_live': ['much live danny'], 'realest_who': ['realest who one']}


=== 2. TF-IDF + Cosine Clustering ===
Topic 'holy_worship': 4 videos
Topic 'mighty_jesus': 2 videos
Topic 'jesus_oh': 1 videos
Topic 'awesome_god': 1 videos
Topic 'aware_nowell': 1 videos
Topic 'grateful_just': 1 videos
Topic 'rhythm_washed': 2 videos
Topic 'forrest_frank': 4 videos
Topic 'courage_kevin': 1 videos
Topic 'come_jesus': 1 videos
Topic 'worthy_deserve': 1 videos
Topic 'trib

In [7]:
# 1. Group playlist dynamically
tfidf_groups = group_playlist_by_tfidf(playlist_videos, distance_threshold=0.7)

# 2. Inspect generated queries
search_queries_dict = generate_topic_search_queries(tfidf_groups)
queries = [q for query_list in search_queries_dict.values() for q in query_list]

print("Generated Clean Queries:", queries)

# 3. Retrieve and deduplicate candidates
raw_candidates = client.search_candidates_by_queries(queries, max_per_query=10)
valid_candidates = filter_existing_playlist_videos(raw_candidates, playlist_videos)

print(f"Raw Candidates: {len(raw_candidates)} | Unique Valid Candidates: {len(valid_candidates)}")

Generated Clean Queries: ['holy website connect', 'boundary power website', 'lips website live', 'power website saved', 'song elevation gospel', 'website live alright', 'elevation website new', 'forrest frank forrestfrank', 'website realized grading', 'power website comejesuscome', 'undeniable website died', 'los gospel live', 'contemporary saved website', 'website always til', 'bibles ain stream', 'rest website alleluiawe', 'believe nazare silas', 'website saved live', 'breath live naomiraineofficial', 'lord 1357347108 song', 'linkfire website once', 'website live written', 'throwing website once', 'website written benji', 'much harder understand', 'shop website freely', 'triumphant power song', 'website father turned', 'website featuring heart', 'live evans ucqfn', 'touches power pay', 'much live danny', 'power leads website', 'thankful 7242515 website', 'power given website', 'realest who one']
Raw Candidates: 50 | Unique Valid Candidates: 47
